In [7]:
import aubio
import numpy as np

def get_chords(wav_file):
    samplerate = 44100
    win_s = 4096  # FFT window size
    hop_s = 512   # Hop size

    s = aubio.source(wav_file, samplerate, hop_s)
    o = aubio.pitch("yin", win_s, hop_s, samplerate)
    o.set_unit("midi")
    o.set_tolerance(0.8)

    pitches = []
    while True:
        samples, read = s()
        pitch = o(samples)[0]
        pitches.append(pitch)
        if read < hop_s:
            break

    return np.array(pitches)

pitches = get_chords("/content/Grand Piano - Fazioli - major A# middle.wav")
print(pitches)


[  0.         0.         0.         0.       139.88118   50.470833
  36.718796  49.95844  160.76558   70.13547   70.017654  70.210785
  70.29527   70.30406   70.60156   72.40206   70.24765   70.24743
  70.26607   70.111115  70.17187   70.19945   70.14926   70.219124
  70.30803   70.26866   70.20502   70.2382    70.19281   70.09952
  70.15603   70.23245   70.15318   70.17332   70.21483   70.10288
  70.131134  70.13713   70.13951   70.17295   70.16517   70.194115
  70.173225  70.161316  70.10458   70.145515  70.175934  70.12475
  70.132996  70.17963   70.02639   69.94265   70.44325   69.98071
  69.70868   71.38422   71.294334  71.39738   71.297424  70.76639
  70.313805  69.89583   70.236595  70.16487   70.09102   70.2022
  70.245804  70.304     70.33694   70.77327   71.30519   71.56749
  58.080383  58.078915  72.559326  58.084633  58.091736  58.098347
  58.10357   70.178535  70.23814   70.285934  58.0921    58.091003
  58.094193  58.09719   58.098114  58.097443  58.095863  58.090218
  58

In [10]:
import numpy as np

# Sample numerical values (from your data)
note_values = pitches

# Trim zeroes
note_values = note_values[note_values > 0]

# Define MIDI note to chord mapping
note_to_chord = {
    (40, 50): "C Major",
    (50, 60): "G Major",
    (60, 70): "D Major",
    (70, 80): "A Minor",
    (80, 90): "E Minor",
    (90, 100): "F Major",
    (100, 110): "B Minor",
    (110, 120): "A Major",
    (120, 130): "E Major",
    (130, 140): "D Minor"
}

# Function to map notes to chords
def map_to_chord(note):
    note = int(note)  # Convert to integer
    for (low, high), chord in note_to_chord.items():
        if low <= note < high:
            return chord
    return "Unknown"

# Convert note values to chords
chord_sequence = [map_to_chord(note) for note in note_values]

# Print the chord sequence
print("Chord Sequence:", " -> ".join(chord_sequence))


Chord Sequence: D Minor -> G Major -> Unknown -> C Major -> Unknown -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> D Major -> A Minor -> D Major -> D Major -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> D Major -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> A Minor -> G Major -> G Major -> A Minor -> G Major -> G Major -> G Major -> G Major -> A Minor -> A Minor -> A Minor -> G Major -> G Major -> G Major -> G Major -> G Major -> G Major -> G Major -> G Major -> G Major -> G Major -> G Major -> G Maj

In [16]:
import os
import aubio
import numpy as np
import pandas as pd

dataset_path = "/content/drive/MyDrive/test_chords"  # Adjust based on actual path

def get_chords(wav_file):
    samplerate = 44100
    win_s = 4096
    hop_s = 512

    s = aubio.source(wav_file, samplerate, hop_s)
    o = aubio.pitch("yin", win_s, hop_s, samplerate)
    o.set_unit("midi")
    o.set_tolerance(0.8)

    pitches = []
    while True:
        samples, read = s()
        pitch = o(samples)[0]
        pitches.append(pitch)
        if read < hop_s:
            break

    return np.array(pitches)

# Process all .wav files
max_length = 300  # Adjust based on observation
all_pitches = []

for file in os.listdir(dataset_path):
    if file.endswith(".wav"):
        file_path = os.path.join(dataset_path, file)
        pitch_seq = get_chords(file_path)

        # Padding or truncating to ensure fixed size
        if len(pitch_seq) < max_length:
            pitch_seq = np.pad(pitch_seq, (0, max_length - len(pitch_seq)), mode='constant')
        else:
            pitch_seq = pitch_seq[:max_length]

        all_pitches.append(pitch_seq)

# Convert to DataFrame
df = pd.DataFrame(all_pitches)
df.to_csv("/content/pitch_dataset.csv", index=False, header=False)

print("Dataset saved as pitch_dataset.csv")


Dataset saved as pitch_dataset.csv


In [81]:
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
import pandas as pd
import numpy as np

# ✅ 1️⃣ Load and Normalize Data
num_features = 300  # Ensure input has 300 features
latent_dim = 16  # Dimension of latent space

# Load dataset (replace with actual dataset)
data = pd.read_csv("/content/pitch_dataset.csv")

# Ensure dataset has correct shape and normalize values
data = data.values  # Convert DataFrame to NumPy array
data = np.nan_to_num(data, nan=0.0)  # Replace NaNs with 0
data = data / np.max(data)  # Normalize to range [0, 1] if needed

# ✅ 2️⃣ Encoder
encoder_inputs = layers.Input(shape=(num_features,))
x = layers.Dense(256, activation="relu")(encoder_inputs)
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)

# ✅ Reparameterization Trick (Fixed)
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = K.random_normal(shape=K.shape(z_mean))
        return z_mean + K.exp(0.5 * z_log_var) * epsilon

z = Sampling()([z_mean, z_log_var])
encoder = models.Model(encoder_inputs, z, name="encoder")  # Only returning `z`

# ✅ 3️⃣ Decoder
latent_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(256, activation="relu")(latent_inputs)
decoder_outputs = layers.Dense(num_features, activation="sigmoid")(x)
decoder = models.Model(latent_inputs, decoder_outputs, name="decoder")

# ✅ 4️⃣ VAE Model
encoded_z = encoder(encoder_inputs)
vae_outputs = decoder(encoded_z)
vae = models.Model(encoder_inputs, vae_outputs, name="vae")

# ✅ 5️⃣ Compile Model with Simple Loss (MSE)
vae.compile(optimizer="adam", loss="mse")

# ✅ 6️⃣ Train the Model
vae.fit(data, data, epochs=10, batch_size=32)

# ✅ 7️⃣ Evaluate Reconstruction Loss
reconstructed_data = vae.predict(data)
final_reconstruction_loss = tf.keras.losses.MeanSquaredError()(data, reconstructed_data).numpy()
print(f"Final Reconstruction Loss: {final_reconstruction_loss:.6f}")


Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 0.0805
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 0.0760
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 0.0725
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 0.0691
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 0.0659
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 0.0608
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0562
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - loss: 0.0509
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 0.0458
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.0421


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Final Reconstruction Loss: 0.038116


In [68]:
data.shape

(23, 300)

In [82]:
import aubio
import numpy as np

def get_chords(wav_file):
    samplerate = 44100
    win_s = 4096  # FFT window size
    hop_s = 512   # Hop size

    s = aubio.source(wav_file, samplerate, hop_s)
    o = aubio.pitch("yin", win_s, hop_s, samplerate)
    o.set_unit("midi")
    o.set_tolerance(0.8)

    pitches = []
    while True:
        samples, read = s()
        pitch = o(samples)[0]
        pitches.append(pitch)
        if read < hop_s:
            break

    return np.array(pitches)

chords = get_chords("/content/Grand Piano - Fazioli - major A# middle.wav")
print(chords)


[  0.         0.         0.         0.       139.88118   50.470833
  36.718796  49.95844  160.76558   70.13547   70.017654  70.210785
  70.29527   70.30406   70.60156   72.40206   70.24765   70.24743
  70.26607   70.111115  70.17187   70.19945   70.14926   70.219124
  70.30803   70.26866   70.20502   70.2382    70.19281   70.09952
  70.15603   70.23245   70.15318   70.17332   70.21483   70.10288
  70.131134  70.13713   70.13951   70.17295   70.16517   70.194115
  70.173225  70.161316  70.10458   70.145515  70.175934  70.12475
  70.132996  70.17963   70.02639   69.94265   70.44325   69.98071
  69.70868   71.38422   71.294334  71.39738   71.297424  70.76639
  70.313805  69.89583   70.236595  70.16487   70.09102   70.2022
  70.245804  70.304     70.33694   70.77327   71.30519   71.56749
  58.080383  58.078915  72.559326  58.084633  58.091736  58.098347
  58.10357   70.178535  70.23814   70.285934  58.0921    58.091003
  58.094193  58.09719   58.098114  58.097443  58.095863  58.090218
  58

In [71]:
chords.shape

(345,)

In [92]:
import aubio
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import soundfile as sf

# 1️⃣ Extract & Normalize Chords from WAV File
def get_chords(wav_file, target_length=300):
    samplerate = 44100
    win_s = 4096  # FFT window size
    hop_s = 512   # Hop size

    s = aubio.source(wav_file, samplerate, hop_s)
    o = aubio.pitch("yin", win_s, hop_s, samplerate)
    o.set_unit("midi")
    o.set_tolerance(0.8)

    pitches = []
    while True:
        samples, read = s()
        pitch = o(samples)[0]  # Extract pitch
        pitches.append(pitch)
        if read < hop_s:
            break

    pitches = np.array(pitches)

    # 2️⃣ Ensure Fixed Input Size (300)
    if len(pitches) < target_length:
        pitches = np.pad(pitches, (0, target_length - len(pitches)), mode='constant')
    else:
        pitches = pitches[:target_length]  # Trim excess

    # 3️⃣ Normalize MIDI Pitch Values to [0, 1]
    pitches = (pitches - pitches.min()) / (pitches.max() - pitches.min() + 1e-6)  # Avoid division by zero

    return pitches.reshape(1, -1)  # Reshape to (1, 300)


# 5️⃣ Load & Preprocess WAV File
wav_path = "/content/Grand Piano - Fazioli - major A# middle.wav"
input_chords = get_chords(wav_path)

# 6️⃣ Generate Reconstructed Chords
reconstructed_chords = vae.predict(input_chords)

# 7️⃣ Compute Reconstruction Loss (MSE)
mse = tf.keras.losses.MeanSquaredError()(input_chords, reconstructed_chords).numpy()
print(f"Reconstruction Loss (MSE): {mse:.6f}")

# 8️⃣ Denormalize the Output (Reverse Scaling)
def denormalize(data, original_min, original_max):
    return data * (original_max - original_min) + original_min

original_min, original_max = 0, 127  # MIDI range
reconstructed_chords = denormalize(reconstructed_chords, original_min, original_max)

# 9️⃣ (Optional) Save Reconstructed Chords to a WAV File
def save_reconstructed_chords(chords, sr=44100):
    sf.write("reconstructed_chords.wav", chords, sr)
    print("Reconstructed chords saved as 'reconstructed_chords.wav'")

save_reconstructed_chords(reconstructed_chords.flatten())


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
Reconstruction Loss (MSE): 0.034182
Reconstructed chords saved as 'reconstructed_chords.wav'


In [93]:
print(reconstructed_chords)

[[ 27.001453  40.24171   32.02845  103.70927   73.95705   93.414116
   61.732742  72.633064  56.428158  50.892517  43.817383  60.287605
   51.122513  66.14841   42.913414  48.735703  57.928246  58.43655
   55.175953  53.013863  43.71216   49.39243   50.838024  51.508907
   41.25189   65.55072   56.188667  57.677883  48.854862  48.649406
   55.508488  38.140945  38.546467  52.277874  53.38649   77.17144
   58.95616   62.89066   43.71156   66.62183   72.13368   45.333786
   45.70274   62.637913  49.925735  65.13525   50.83918   54.627747
   46.428402  59.20271   50.79919   57.99361   41.87068   53.536064
   69.42702   53.87507   54.999264  64.43046   48.435905  61.92812
   52.872967  54.86606   62.91458   63.98761   61.095104  62.36329
   64.31358   64.38182   44.93724   52.60971   53.862713  54.38208
   55.6517    47.852295  60.67593   54.883816  63.883305  55.097862
   61.80538   65.23346   63.335846  42.863873  49.783367  45.546864
   54.90807   74.3311    45.11198   66.956154  69.196

In [94]:
chords.shape

(300,)